In [1]:
# 04_guards.ipynb
# ------------------------------------------------------------------------------------
# Imports, configuration, grayscale plotting style, and output savers.
# Design choices: single-cell notebook; English comments only; figures in grayscale via
# seaborn; no captions; dpi=600; every figure saved as BOTH .png and .pdf; table outputs
# saved as .csv; file paths are never printed (anonymity).
# ------------------------------------------------------------------------------------
import os, copy, hashlib, warnings
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

# Reproducibility
GLOBAL_SEED = 2025
np.random.seed(GLOBAL_SEED)

# Paths (relative; intentionally never printed)
DATA_PATH   = os.path.join("..", "data", "criteo_clean.parquet")
FIG_DIR     = os.path.join("..", "results", "figures")
TBL_DIR     = os.path.join("..", "results", "tables")
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TBL_DIR, exist_ok=True)

# Grayscale, publication-style seaborn theme
sns.set_theme(context="paper", style="whitegrid")
GRAYSCALE = ["#111111", "#555555", "#999999", "#cccccc"]
sns.set_palette(sns.color_palette(GRAYSCALE))
plt.rcParams.update({
    "image.cmap": "gray",
    "axes.edgecolor": "#333333",
    "axes.labelcolor": "#000000",
    "text.color": "#000000",
    "xtick.color": "#000000",
    "ytick.color": "#000000",
    "font.size": 11,
    "savefig.bbox": "tight",
})

def save_fig(fig, name):
    """Save a figure as png and pdf at dpi=600. No captions, no path printing."""
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG_DIR, f"{name}.{ext}"), dpi=600)
    plt.close(fig)

def save_table(df, name):
    """Save a results table as csv. No path printing."""
    df.to_csv(os.path.join(TBL_DIR, f"{name}.csv"), index=False)

# Experiment configuration
CFG = {
    "data": {
        "clean_path": DATA_PATH,
        # Only features observable at BIDDING TIME are used. The Criteo Attribution dataset's
        # click_pos/click_nb/cost/cpo/attribution are computed along the conversion PATH (e.g.
        # click_pos = "position of the click before a conversion", cpo = cost per ORDER), so they
        # are post-conversion and leak the label (each gives standalone AUROC ~1.0 vs conversion).
        # `click` is the stage-1 LABEL and must never be an input. We keep contextual categoricals,
        # the campaign id, and time_since_last_click (past-only recency).
        "cat_cols": ["campaign","cat1","cat2","cat3","cat4","cat5","cat6","cat7","cat8","cat9"],
        "dense_cols": ["time_since_last_click"],
        "stage_cols": ["click","conversion"],
        "max_vocab": 100000,
    },
    "split": {"use_existing_split": True, "train_max_day": 20, "valid_max_day": 24},
    "model": {"name":"leakfunnel","embed_dim":16,"hidden_dim":128,"n_heads":4,"n_layers":2,
              "dropout":0.1,"lambda_mono":0.1,"lambda_leak":0.1},
    # num_workers MUST be 0 on Windows + single-cell notebooks: spawned workers cannot
    # re-import classes defined in the notebook and will deadlock on the first batch.
    # We avoid DataLoader workers entirely by moving each split to the GPU up front
    # (see _tensors) and batching by index, so num_workers is not used for training.
    # amp (float16 mixed precision) is OFF by default: these models are small enough to be
    # fast in float32 on a consumer GPU, and AMP can produce NaN predictions on the
    # Transformer/ESMM paths. Turn it on only if you need the speed and have verified stability.
    "train": {"epochs":3,"batch_size":8192,"lr":1e-3,"weight_decay":1e-5,"num_workers":0,
              "device":"auto","seed":GLOBAL_SEED,"amp":False,"max_rows":None},
    "leakage": {"as_of_days":[15,20,25],"leak_strength":0.5,"leak_fraction":0.5,
                "noise_sd":0.3,"family":"both"},
    "eval": {"n_seeds":5,"bootstrap":1000},
}

# FAST switch: set FAST = True for a quick pass (subsampled rows, fewer epochs/seeds) to
# iterate or sanity-check; set FAST = False for the full, paper-grade run. One line to flip.
FAST = False
if FAST:
    CFG["train"]["max_rows"] = 300000
    CFG["train"]["epochs"] = 2
    CFG["eval"]["n_seeds"] = 2

def audit_input_leakage(cfg, label_col="conversion", thresh=0.99, n=300000):
    # Self-check: measure each MODEL INPUT feature's standalone AUROC against the label on a
    # subsample. A value near 1.0 means the feature essentially encodes the label (a
    # post-outcome/leaky column that must not be an input). Prints a warning per offender.
    # This is a guard against silently re-introducing label leakage into the baseline pipeline.
    from sklearn.metrics import roc_auc_score
    cols = cfg["data"]["cat_cols"] + cfg["data"]["dense_cols"] + [label_col]
    t = pq.read_table(cfg["data"]["clean_path"], columns=list(dict.fromkeys(cols)))
    m = t.num_rows
    idx = np.linspace(0, m-1, min(n, m)).astype(np.int64)
    df = t.take(idx).to_pandas()
    y = df[label_col].to_numpy()
    flagged = []
    for c in cfg["data"]["cat_cols"] + cfg["data"]["dense_cols"]:
        x = np.nan_to_num(df[c].to_numpy().astype(float))
        try:
            a = roc_auc_score(y, x); a = max(a, 1 - a)
        except Exception:
            a = float("nan")
        if np.isfinite(a) and a >= thresh:
            flagged.append((c, a)); print(f"LEAKAGE WARNING: input '{c}' has standalone AUROC {a:.4f} vs {label_col} -> remove it")
    if not flagged:
        print(f"input leakage audit: OK (no input feature exceeds AUROC {thresh} vs {label_col})")
    return flagged

def report_device():
    # Print the compute device once so it is obvious whether CUDA is actually used.
    # If this shows CPU on a GPU machine, the installed torch is a CPU build -> reinstall
    # torch with the CUDA index URL (see README). No file paths are printed.
    try:
        import torch
        if torch.cuda.is_available():
            print(f"device: cuda ({torch.cuda.get_device_name(0)}) | torch {torch.__version__}")
        else:
            print(f"device: cpu | torch {torch.__version__} | CUDA not available")
    except ImportError:
        print("torch not installed (only data/detection notebooks run without it)")

# ------------------------------------------------------------------------------------
# RQ6: deployment guards. Feature pinning (schema-hash divergence) and degradation
# gating (per-feature null fraction). Firing rates and false-alarm rate under simulated
# schema drift and null floods.
# ------------------------------------------------------------------------------------
cols = CFG["data"]["cat_cols"] + CFG["data"]["dense_cols"]
trained_hash = hashlib.sha256("|".join(sorted(cols)).encode()).hexdigest()

rng = np.random.default_rng(GLOBAL_SEED)
n_trials = 1000
null_threshold = 0.3
pin_fires = 0; gate_fires = 0; false_alarms = 0
pin_series = []; gate_series = []
for _ in range(n_trials):
    live = list(cols)
    drift = rng.random() < 0.5
    if drift and len(live) > 1:
        live.pop(rng.integers(len(live)))
    live_hash = hashlib.sha256("|".join(sorted(live)).encode()).hexdigest()
    pin = live_hash != trained_hash
    null_frac = rng.random() if rng.random() < 0.5 else rng.random() * 0.1
    gate = null_frac > null_threshold
    pin_fires += int(pin); gate_fires += int(gate)
    if (not drift) and pin: false_alarms += 1
    pin_series.append(int(pin)); gate_series.append(int(gate))

rq6 = pd.DataFrame([{
    "n_trials": n_trials,
    "pinning_fire_rate": pin_fires / n_trials,
    "gating_fire_rate": gate_fires / n_trials,
    "pinning_false_alarm_rate": false_alarms / n_trials,
    "gating_threshold": null_threshold,
}])
save_table(rq6, "rq6_guards")

fig, ax = plt.subplots(figsize=(6, 4))
labels = ["feature pinning", "degradation gating"]
rates = [pin_fires / n_trials, gate_fires / n_trials]
ax.bar(labels, rates, color=["#333333", "#999999"], edgecolor="#000000")
ax.set_ylabel("firing rate"); ax.set_ylim(0, 1)
save_fig(fig, "rq6_guard_firing_rates")

rq6


,n_trials,pinning_fire_rate,gating_fire_rate,pinning_false_alarm_rate,gating_threshold
0,1000,0.489,0.352,0.0,0.3
